## Problem Statement

Develop a machine learning model that predicts an individual's annual medical insurance charges based on demographic and lifestyle factors such as age, BMI, smoking status, gender, number of children, and region. The objective is to help insurance companies estimate premiums more accurately and assist customers in understanding the factors influencing healthcare costs.

# OverView

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib


In [ ]:
dataset = pd.read_csv("../data/raw/insurance.csv")

In [ ]:
dataset.head()   

In [ ]:
dataset.info()

In [ ]:
dataset.describe()

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.duplicated().sum()

In [ ]:
dataset = dataset.drop_duplicates()

# Identify feature types

In [ ]:
target = "charges"

X=dataset.drop('charges',axis=1)
Y=dataset[target]

In [ ]:
cat_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_features = X.select_dtypes(include=["number"]).columns.tolist()

print("Categorical:", cat_features)
print("Numerical:", num_features)

# EDA

Distribution of Numerical Features

In [ ]:
dataset.hist(figsize=(12,8))
plt.tight_layout()
plt.show()

Distribution of Categorical Features

In [ ]:
for col in cat_features:
    plt.figure(figsize=(5,4))
    sns.countplot(data=dataset, x=col)
    plt.title(col)
    plt.show()

Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(dataset.corr(numeric_only=True),
            annot=True,
            cmap="coolwarm")
plt.show()

Outlier Detection

In [ ]:
for col in num_features:
    plt.figure(figsize=(5,3))
    sns.boxplot(x=dataset[col])
    plt.title(col)
    plt.show()

Charges Distribution

In [ ]:
sns.histplot(dataset["charges"], kde=True)
plt.show()

Relationship with Target

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(data=dataset, x="age", y="charges", ax=axes[0,0])
axes[0,0].set_title("Age vs Charges")

sns.scatterplot(data=dataset, x="bmi", y="charges", ax=axes[0,1])
axes[0,1].set_title("BMI vs Charges")

sns.boxplot(data=dataset, x="smoker", y="charges", ax=axes[1,0])
axes[1,0].set_title("Smoker vs Charges")

sns.boxplot(data=dataset, x="sex", y="charges", ax=axes[1,1])
axes[1,1].set_title("Sex vs Charges")

plt.tight_layout()
plt.show()

# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.25,
    random_state=42
)

## Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),num_features),
        ("cat",OneHotEncoder(),cat_features)
    ]
)

# Linear Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

linear = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear.fit(X_train, y_train)

y_pred = linear.predict(X_test)

# Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuals)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()

In [ ]:
joblib.dump(linear, "../models/linear_regression_pipeline.pkl")

## Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)
y_pred = ridge_grid.predict(X_test)


In [ ]:
print(ridge_grid.best_params_)
print(ridge_grid.best_score_)

Evaluation

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuals)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()

In [ ]:
joblib.dump(
    ridge_grid.best_estimator_,
    "../models/ridge_pipeline.pkl"
)

## Lasso Regression

In [ ]:
lasso_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(max_iter=10000))
])

param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]
}

lasso_grid = GridSearchCV(
    lasso_pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

lasso_grid.fit(X_train, y_train)
y_pred = lasso_grid.predict(X_test)


In [ ]:
print(lasso_grid.best_params_)
print(lasso_grid.best_score_)

Evaluation

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuals)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()

In [ ]:
joblib.dump(
    lasso_grid.best_estimator_,
    "../models/lasso_pipeline.pkl"
)

# Elaticnet Regression

In [ ]:
elastic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ElasticNet(max_iter=10000))
])

param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1],
    "model__l1_ratio": [0.2, 0.4, 0.6, 0.8, 1.0]
}

elastic_grid = GridSearchCV(
    elastic_pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

elastic_grid.fit(X_train, y_train)
y_pred = elastic_grid.predict(X_test)



In [ ]:
print(elastic_grid.best_params_)
print(elastic_grid.best_score_)

In [ ]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuals)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Charges")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()

In [ ]:
joblib.dump(
    elastic_grid.best_estimator_,
    "../models/elasticnet_pipeline.pkl"
)

# Model Comparison

In [ ]:
import json

results = {
    "Linear Regression": linear.predict(X_test),
    "Ridge": ridge_grid.best_estimator_.predict(X_test),
    "Lasso": lasso_grid.best_estimator_.predict(X_test),
    "ElasticNet": elastic_grid.best_estimator_.predict(X_test),
}

comparison = []
for name, preds in results.items():
    comparison.append({
        "Model": name,
        "R2": r2_score(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "MAE": mean_absolute_error(y_test, preds)
    })

comparison_df = pd.DataFrame(comparison).sort_values("R2", ascending=False).reset_index(drop=True)
comparison_df


Best model based on R² score is selected below. Its metrics are saved to `metrics.json` so the Streamlit app can load them dynamically instead of using hardcoded values.

In [ ]:
model_files = {
    "Linear Regression": "linear_regression_pipeline.pkl",
    "Ridge": "ridge_pipeline.pkl",
    "Lasso": "lasso_pipeline.pkl",
    "ElasticNet": "elasticnet_pipeline.pkl"
}

all_metrics = {}
for row in comparison_df.itertuples():
    all_metrics[row.Model] = {
        "file": model_files[row.Model],
        "r2": f"{row.R2:.3f}",
        "rmse": f"{row.RMSE:.0f}",
        "mae": f"{row.MAE:.0f}"
    }

best_model_name = comparison_df.iloc[0]["Model"]

metrics = {
    "best_model": best_model_name,
    "models": all_metrics
}

with open("../models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Best model:", best_model_name)
print(json.dumps(metrics, indent=2))

In [ ]:
<p align="center">
  <img src="images/app_preview.png" alt="Insurance Premium Prediction App" width="900">
</p>
